Download all molecules that have been tested against the target of interest - epidermal growth factor receptor (EGFR) kinase.

In [34]:
import math
from pathlib import Path
from zipfile import ZipFile
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd
from rdkit.Chem import PandasTools
from chembl_webresource_client.new_client import new_client
import tqdm as notebook_tqdm

In [35]:
last_dir = Path(_dh[-1])
print(f"Current working directory: {last_dir}")

Current working directory: /Users/duncanfreeman/dev/bioinfo/notebooks/volkmer


In [36]:
targets_api = new_client.target
compounds_api = new_client.molecule
bioactivities_api = new_client.activity

In [37]:
uniprot_id = "P00533"
# Get target information from ChEMBL but restrict it to specified values only
targets = targets_api.get(target_components__accession=uniprot_id).only(
    "target_chembl_id", "organism", "pref_name", "target_type"
)
print(f'The type of the targets is "{type(targets)}"')
targets = pd.DataFrame.from_records(targets)
targets

The type of the targets is "<class 'chembl_webresource_client.query_set.QuerySet'>"


,organism,pref_name,target_chembl_id,target_type
0,Homo sapiens,Epidermal growth factor receptor,CHEMBL203,SINGLE PROTEIN
1,Homo sapiens,Epidermal growth factor receptor,CHEMBL203,SINGLE PROTEIN
2,Homo sapiens,Epidermal growth factor receptor and ErbB2 (HE...,CHEMBL2111431,PROTEIN FAMILY
3,Homo sapiens,Epidermal growth factor receptor,CHEMBL2363049,PROTEIN FAMILY
4,Homo sapiens,MER intracellular domain/EGFR extracellular do...,CHEMBL3137284,CHIMERIC PROTEIN
5,Homo sapiens,Protein cereblon/Epidermal growth factor receptor,CHEMBL4523680,PROTEIN-PROTEIN INTERACTION
6,Homo sapiens,EGFR/PPP1CA,CHEMBL4523747,PROTEIN-PROTEIN INTERACTION
7,Homo sapiens,von Hippel-Lindau disease tumor suppressor/Epi...,CHEMBL4523998,PROTEIN-PROTEIN INTERACTION
8,Homo sapiens,Baculoviral IAP repeat-containing protein 2/Ep...,CHEMBL4802031,PROTEIN-PROTEIN INTERACTION
9,Homo sapiens,CCN2-EGFR,CHEMBL5465557,PROTEIN-PROTEIN INTERACTION


In [38]:
target = targets.iloc[0]
target

organism                                Homo sapiens
pref_name           Epidermal growth factor receptor
target_chembl_id                           CHEMBL203
target_type                           SINGLE PROTEIN
Name: 0, dtype: str

In [ ]:
# save selected ChEMBL ID to variable for later use
chembl_id = target.target_chembl_id
print(f"The target ChEMBL ID is {chembl_id}")
# NBVAL_CHECK_OUTPUT

The target ChEMBL ID is CHEMBL203


### Get Bioactivity Data
To query bioactivity data for the target of interest:

Fetch bioactivity data for the target from ChEMBL¶
In this step, we fetch the bioactivity data and filter it to only consider

- human proteins
- bioactivity type IC50
- exact measurements (relation '=')
- binding data (assay type 'B')

In [41]:
bioactivities = bioactivities_api.filter(
    target_chembl_id=chembl_id, type="IC50", relation="=", assay_type="B"
).only(
    "activity_id",
    "assay_chembl_id",
    "assay_description",
    "assay_type",
    "molecule_chembl_id",
    "type",
    "standard_units",
    "relation",
    "standard_value",
    "target_chembl_id",
    "target_organism",
)

print(f"Length and type of bioactivities object: {len(bioactivities)}, {type(bioactivities)}")

Length and type of bioactivities object: 17686, <class 'chembl_webresource_client.query_set.QuerySet'>


In [42]:
print(f"Length and type of first element: {len(bioactivities[0])}, {type(bioactivities[0])}")
bioactivities[0]

Length and type of first element: 13, <class 'dict'>


{'activity_id': 32260,
 'assay_chembl_id': 'CHEMBL674637',
 'assay_description': 'Inhibitory activity towards tyrosine phosphorylation for the epidermal growth factor-receptor kinase',
 'assay_type': 'B',
 'molecule_chembl_id': 'CHEMBL68920',
 'relation': '=',
 'standard_units': 'nM',
 'standard_value': '41.0',
 'target_chembl_id': 'CHEMBL203',
 'target_organism': 'Homo sapiens',
 'type': 'IC50',
 'units': 'uM',
 'value': '0.041'}

### Download bioactivity data from ChEMBL

In [68]:
import os
import tarfile
import urllib.request
import sqlite3
import pandas as pd
from tqdm import tqdm

# 1. Define your custom paths
TARGET_DIR_DB = "/Users/duncanfreeman/dev/bioinfo/data/"  
CHEMBL_VERSION = "37" 
TAR_FILENAME = f"chembl_{CHEMBL_VERSION}_sqlite.tar.gz"                       # Update version number if needed

# Setup exact file paths
tar_gz_filename = f"chembl_{CHEMBL_VERSION}_sqlite.tar.gz"
download_url = f"https://ebi.ac.uk{tar_gz_filename}"
local_tar_path = os.path.join(TARGET_DIR_DB, tar_gz_filename)
extracted_db_path = os.path.join(TARGET_DIR_DB, f"chembl_{CHEMBL_VERSION}", f"chembl_{CHEMBL_VERSION}.db")

os.makedirs(TARGET_DIR_DB, exist_ok=True)

# 2. Helper class to show a visual progress bar during download
class TqdmUpTo(tqdm):
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)

# 3. Download the archive if it doesn't exist
if not os.path.exists(extracted_db_path):
    if not os.path.exists(local_tar_path):
        print(f"Downloading ChEMBL v{CHEMBL_VERSION} to {local_tar_path}...")
        with TqdmUpTo(unit='B', unit_scale=True, unit_divisor=1024, miniters=1, desc="Downloading") as t:
            urllib.request.urlretrieve(download_url, filename=local_tar_path, reporthook=t.update_to)
    
    # 4. Extract the database file
    print(f"Extracting archive inside {TARGET_DIR_DB}...")
    with tarfile.open(local_tar_path, "r:gz") as tar:
        tar.extractall(path=TARGET_DIR_DB)
        
    # Clean up the large tar.gz file after successful extraction to save disk space
    os.remove(local_tar_path)
    print("Extraction complete. Cleaned up archive file.")
else:
    print(f"Database already exists at: {extracted_db_path}")

# 5. Connect and query using Pandas (Same high-performance query)
print("Connecting to local SQLite database...")
conn = sqlite3.connect(extracted_db_path)

target_id = "CHEMBL240" 
sql_query = """
SELECT 
    act.activity_id, ass.chembl_id AS assay_chembl_id, ass.description AS assay_description,
    ass.assay_type, md.chembl_id AS molecule_chembl_id, act.type, act.standard_units,
    act.relation, act.standard_value, trg.chembl_id AS target_chembl_id, trg.organism AS target_organism
FROM activities act
JOIN assays ass ON act.assay_id = ass.assay_id
JOIN target_dictionary trg ON ass.tid = trg.tid
JOIN molecule_dictionary md ON act.molregno = md.molregno
WHERE trg.chembl_id = ? AND act.type = 'IC50' AND act.relation = '=' AND ass.assay_type = 'B';
"""

bioactivities_df = pd.read_sql_query(sql_query, conn, params=(target_id,))
conn.close()

print(f"\nSuccess! Rows fetched: {len(bioactivities_df)}")


Downloading: 0.00B [00:00, ?B/s]


URLError: <urlopen error [Errno 8] nodename nor servname provided, or not known>

In [66]:
import time
from chembl_webresource_client.http_errors import HttpApplicationError

# 1. Configuration for robust batch pulling
CHUNK_SIZE = 100       # Smaller chunks prevent ChEMBL from timing out
MAX_RETRIES = 5        # Number of times to retry a failed chunk
BACKOFF_FACTOR = 2     # Wait 2s, 4s, 8s... between retries

all_records = []
total_records = len(bioactivities) # bioactivities is a generator, so we need to evaluate it to get the total count
print(f"Total records to fetch: {total_records}")

# 2. Paginated loop with defensive error handling
for offset in range(0, total_records, CHUNK_SIZE):
    limit = offset + CHUNK_SIZE
    retries = 0
    success = False
    
    while not success and retries < MAX_RETRIES:
        try:
            # Force evaluation of a small slice over the network
            chunk = list(bioactivities[offset:limit])
            all_records.extend(chunk)
            success = True
            print(f"Successfully fetched rows {offset} to {min(limit, total_records)}")
            
        except (HttpApplicationError, Exception):
            retries += 1
            wait_time = BACKOFF_FACTOR ** retries
            print(f"Error at rows {offset}-{limit}. Retrying ({retries}/{MAX_RETRIES}) in {wait_time}s...")
            time.sleep(wait_time)
            
    if not success:
        print(f"CRITICAL: Failed to fetch chunk starting at offset {offset} after {MAX_RETRIES} attempts.")
        # Optional: break or choose to keep whatever data was successfully pulled up to this point
        break

# 4. Safely load into a DataFrame
bioactivities_query_df = pd.DataFrame(all_records)
print(f"\nFinal DataFrame shape: {bioactivities_query_df.shape}")

Total records to fetch: 17686
Successfully fetched rows 0 to 100
Successfully fetched rows 100 to 200
Successfully fetched rows 200 to 300
Successfully fetched rows 300 to 400
Successfully fetched rows 400 to 500
Successfully fetched rows 500 to 600
Successfully fetched rows 600 to 700
Successfully fetched rows 700 to 800
Successfully fetched rows 800 to 900
Successfully fetched rows 900 to 1000
Successfully fetched rows 1000 to 1100
Successfully fetched rows 1100 to 1200
Successfully fetched rows 1200 to 1300
Successfully fetched rows 1300 to 1400
Successfully fetched rows 1400 to 1500
Successfully fetched rows 1500 to 1600
Successfully fetched rows 1600 to 1700
Successfully fetched rows 1700 to 1800
Successfully fetched rows 1800 to 1900
Successfully fetched rows 1900 to 2000
Successfully fetched rows 2000 to 2100
Successfully fetched rows 2100 to 2200
Successfully fetched rows 2200 to 2300
Successfully fetched rows 2300 to 2400
Successfully fetched rows 2400 to 2500
Successfully fet

KeyboardInterrupt: 

In [52]:
bioactivities_df = pd.DataFrame.from_dict(bioactivities)
print(f"DataFrame shape: {bioactivities_df.shape}")
bioactivities_df.head(2)

HttpApplicationError: Error for url https://www.ebi.ac.uk/chembl/api/data/activity.json, server response: <!doctype html>
<html lang="en" class="vf-no-js">
  <head>
    <script>
// Detect if JS is on and swap vf-no-js for vf-js on the html element
(function(H){H.className=H.className.replace(/\bvf-no-js\b/,'vf-js')})(document.documentElement);
</script>

    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <!-- <link rel="stylesheet" media="all" href="/css/styles.css?" /> -->
    <title>Error: 500 | EMBLâs European Bionformatics Institute</title>



    <link rel="icon" type="image/x-icon"
  href="https://ebi.emblstatic.net/web_guidelines/EBI-Framework/v1.4/images/logos/EMBL-EBI/favicons/favicon.ico" />
<link rel="icon" type="image/png"
  href="https://ebi.emblstatic.net/web_guidelines/EBI-Framework/v1.4/images/logos/EMBL-EBI/favicons/favicon-32x32.png" />
<link rel="icon" type="image/png" sizes="192Ã192"
  href="https://ebi.embls

The first two rows describe the same bioactivity entry; we will remove such artifacts later during the deduplication step. Note also that we have columns for standard_units/units and standard_values/values; in the following, we will use the standardized columns (standardization by ChEMBL), and thus, we drop the other two columns.

If we used the units and values columns, we would need to convert all values with many different units to nM:



In [ ]:
bioactivities_df["units"].unique()

In [ ]:
bioactivities_df.drop(["units", "value"], axis=1, inplace=True)
bioactivities_df.head()

In [ ]:
bioactivities_df = bioactivities_df.astype({"standard_value": "float64"})
bioactivities_df.dtypes

In [ ]:
bioactivities_df.dropna(axis=0, how="any", inplace=True)
print(f"DataFrame shape: {bioactivities_df.shape}")
print(f"Number of non-nM entries: {bioactivities_df[bioactivities_df['standard_units'] != 'nM'].shape[0]}")
print(f"DataFrame shape: {bioactivities_df.shape}")

In [ ]:
bioactivities_df = bioactivities_df[bioactivities_df["standard_units"] == "nM"]
print(f"Units after filtering: {bioactivities_df['standard_units'].unique()}")

In [ ]:
bioactivities_df.drop_duplicates("molecule_chembl_id", keep="first", inplace=True)
print(f"DataFrame shape: {bioactivities_df.shape}")

In [ ]:
bioactivities_df.reset_index(drop=True, inplace=True)
bioactivities_df.rename(
    columns={"standard_value": "IC50", "standard_units": "units"}, inplace=True
)
bioactivities_df.head(2)

In [ ]:
print(f"Number of unique molecules: {bioactivities_df['molecule_chembl_id'].nunique()}")
print(f"Number of unique assays: {bioactivities_df['assay_chembl_id'].nunique()}")
print(f"Number of unique activities: {bioactivities_df['activity_id'].nunique()}")
print(f"Number of unique targets: {bioactivities_df['target_chembl_id'].nunique()}")

Dataframe contains all molecules tested against EGFR.
Retrieve the molecular structures of the molecules that are linked to respective bioactivity ChEMBL IDs.

Fetch compound data from ChEMBL¶
Let’s have a look at the compounds from ChEMBL which we have defined bioactivity data for: 
- Fetch compound ChEMBL IDs and structures for the compounds linked to our filtered bioactivity data.

In [ ]:
compounds_provider = compounds_api.filter(
    molecule_chembl_id__in=list(bioactivities_df["molecule_chembl_id"])
).only("molecule_chembl_id", "molecule_structures")

In [ ]:
import time

molecule_ids = (
    bioactivities_df["molecule_chembl_id"]
    .dropna()
    .drop_duplicates()
    .astype(str)
    .tolist()
)

print(f"Fetching structures for {len(molecule_ids)} molecules")

def fetch_compounds(ids, chunk_size=100):
    out = []
    for start in range(0, len(ids), chunk_size):
        chunk = ids[start:start + chunk_size]
        for attempt in range(5):
            try:
                batch = list(
                    compounds_api.filter(molecule_chembl_id__in=chunk).only(
                        "molecule_chembl_id", "molecule_structures"
                    )
                )
                out.extend(batch)
                break
            except Exception as e:
                if attempt == 4:
                    raise
                print(f"Retry {attempt + 1} for chunk {start}-{start + len(chunk)}: {e}")
                time.sleep(2 ** attempt)
    return out

compounds = fetch_compounds(molecule_ids)
print(f"Downloaded {len(compounds)} compound records")

In [ ]:
from tqdm import tqdm
compounds = list(tqdm((compounds_provider)))